## Backstage-Container in Kubernetes starten

Minimaler Ablauf: Namespace erstellen, Secrets anlegen, Image als Deployment starten und über einen Service erreichbar machen.

### Namespace erstellen

Die Backstage-Ressourcen werden in einem eigenen Kubernetes Namespace abgelegt.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Namespace
metadata:
  name: backstage
EOF

### Secrets in Kubernetes anlegen

Die Zugangsdaten werden als Kubernetes Secret gespeichert und später als Umgebungsvariablen in den Container geladen.

In [ ]:
%%bash
set -a
source ~/data/env-platen.py
export BACKSTAGE_HOSTNAME="$(cat ~/data/server-ip)"
export BACKSTAGE_NAME="Backstage Produktion"
export BACKSTAGE_PORT="7007"

kubectl apply -f - <<EOF
apiVersion: v1
kind: Secret
metadata:
  name: backstage-secrets
  namespace: backstage
type: Opaque
stringData:
  GITHUB_TOKEN: "${GITHUB_TOKEN}"
  GITLAB_TOKEN: "${GITLAB_TOKEN}"
  BACKSTAGE_HOST: "${BACKSTAGE_HOSTNAME}"
  BACKSTAGE_NAME: "${BACKSTAGE_NAME}"
  BACKSTAGE_PORT: "${BACKSTAGE_PORT}"
EOF

### Backstage Deployment erstellen

Das Deployment startet das vorbereitete Backstage-Image. Alle Werte aus `backstage-secrets` werden als Umgebungsvariablen übernommen.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: apps/v1
kind: Deployment
metadata:
  name: backstage
  namespace: backstage
spec:
  replicas: 1
  selector:
    matchLabels:
      app: backstage
  template:
    metadata:
      labels:
        app: backstage
    spec:
      containers:
        - name: backstage
          image: registry.gitlab.com/ch-mc-b/autoshop-ms/infra/backstage/backstage:1.51
          imagePullPolicy: IfNotPresent
          ports:
            - name: http
              containerPort: 7007
          envFrom:
            - secretRef:
                name: backstage-secrets
EOF

### Service erstellen

Der NodePort-Service veröffentlicht Backstage auf Port `30077` jedes Kubernetes-Nodes.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: Service
metadata:
  name: backstage
  namespace: backstage
spec:
  type: NodePort
  selector:
    app: backstage
  ports:
    - name: http
      port: 7007
      targetPort: 7007
      nodePort: 30007
EOF

### Backstage öffnen


In [ ]:

%%bash
echo "URL: http://$(cat ~/data/server-ip):30007"

- - --
### Aufräumen

In [ ]:
%%bash
kubectl delete --namespace backstage secret/backstage-secrets || true
kubectl delete --namespace backstage deployment/backstage || true
kubectl delete --namespace backstage service/backstage || true
kubectl delete namespace backstage || true